In [ ]:
import json
import pathlib

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from jax import numpy as jnp

from vizopt.animation import SnapshotCallback
from vizopt.base import OptimConfig
from vizopt.schedules import make_term_schedules
from vizopt.templates.stars_vs_circles import (
    optimize_multiple_radially_convex_sets,
    optimize_multiple_radially_convex_sets_with_movable_circles,
)

## Languages

In [ ]:

graph_data_path = pathlib.Path().resolve().parent / "data" / "glotto_subgraph.json"

with open(graph_data_path, "r") as f:
    graph_data = json.load(f)
glotto_graph = nx.node_link_graph(graph_data)
print(f"Loaded {len(glotto_graph.nodes)} nodes and {len(glotto_graph.edges)} edges")
glotto_graph

In [ ]:
leaves = [n for n in glotto_graph.nodes() if glotto_graph.out_degree(n) == 0]
for n in leaves:
    if glotto_graph.nodes[n].get("longitude") is None:
        print(n, glotto_graph.nodes[n], nx.ancestors(glotto_graph, n))

In [ ]:


# ── Build circles (leaves) ────────────────────────────────────────────────────
leaves = [n for n in glotto_graph.nodes() if glotto_graph.out_degree(n) == 0]
non_leaves = [n for n in glotto_graph.nodes() if glotto_graph.out_degree(n) > 0]
leaf_index = {n: i for i, n in enumerate(leaves)}

# Collect lat/lon from leaves that have them; use global centroid as fallback
lons = [glotto_graph.nodes[n]["longitude"] for n in leaves if glotto_graph.nodes[n].get("longitude") is not None]
lats = [glotto_graph.nodes[n]["latitude"]  for n in leaves if glotto_graph.nodes[n].get("latitude")  is not None]
mean_lon, mean_lat = np.mean(lons), np.mean(lats)
lon_scale_factor = 10/(np.max(lons) - np.min(lons))
lat_scale_factor = 10/(np.max(lats) - np.min(lats))

circles = []
for n in leaves:
    d = glotto_graph.nodes[n]
    x = lon_scale_factor*(d["longitude"] if d.get("longitude") is not None else mean_lon)
    y = lat_scale_factor*(d["latitude"]  if d.get("latitude")  is not None else mean_lat)
    r = d.get("size", 0.4)
    circles.append((x, y, r))

# ── Build sets (all leaf-descendants of each non-leaf) ───────────────────────
sets = []
set_names = []
set_nodes = []
for n in non_leaves:
    desc_leaves = [v for v in nx.descendants(glotto_graph, n) if glotto_graph.out_degree(v) == 0]
    if desc_leaves:
        sets.append([leaf_index[l] for l in desc_leaves])
        set_names.append(glotto_graph.nodes[n]["name"])
        set_nodes.append(n)

print(f"{len(circles)} circles, {len(sets)} sets")

# ── Compute offsets: unit_offset * path length from set node to each leaf ────────────
unit_offset = 0.2
glotto_undirected = glotto_graph.to_undirected()
root_nodes = [node for node in glotto_graph if glotto_graph.in_degree(node) == 0]
main_root = "eve"
glotto_undirected.add_node(main_root)
for node in root_nodes:
    glotto_undirected.add_edge(main_root, node)

S, N = len(sets), len(leaves)
offsets = np.zeros((S, N), dtype=np.float32)
for s, set_node in enumerate(set_nodes):
    for n, leaf in enumerate(leaves):
        try:
            path_len = nx.shortest_path_length(glotto_undirected, set_node, leaf)
        except nx.NetworkXNoPath:
            print(f"Warning: no path between {set_node} and {leaf}, this should not happen")
            path_len = 1
        offsets[s, n] = unit_offset * path_len

In [ ]:
# ── Snap geographically displaced nodes to their linguistic family centroid ───
# Some languages (e.g. Afrikaans) are spoken far from their closest relatives.
# Starting them at their geographic position traps the optimizer in a local
# minimum where intervening language groups block the path to the right cluster.
# We snap any leaf whose scaled position is more than `threshold` units from
# its family centroid to that centroid instead.
#
# "Family centroid" walks up the ancestor chain until it finds at least
# `min_peers` other leaves — small groups like {Dutch, Afrikaans} don't provide
# a stable reference, but West Germanic with many Northern-European leaves does.
threshold = 3.0  # scaled coordinate units (total span is 10 per axis)
min_peers = 3

original_xy = np.array([(cx, cy) for cx, cy, _ in circles])  # fixed reference

def _family_centroid(node, min_peers):
    """Walk up the ancestor chain; return centroid of first group with >= min_peers peers."""
    current = node
    while True:
        parents = list(glotto_graph.predecessors(current))
        if not parents:
            return None
        current = parents[0]
        peers = [
            v for v in nx.descendants(glotto_graph, current)
            if glotto_graph.out_degree(v) == 0 and v != node
        ]
        if len(peers) >= min_peers:
            return original_xy[[leaf_index[p] for p in peers]].mean(axis=0)

circles_list = list(circles)
for i, n in enumerate(leaves):
    centroid = _family_centroid(n, min_peers)
    if centroid is None:
        continue
    cx, cy, r = circles_list[i]
    if np.hypot(cx - centroid[0], cy - centroid[1]) > threshold:
        print(f"Snapping {glotto_graph.nodes[n]['name']:20s}  "
              f"({cx:+.2f}, {cy:+.2f}) → ({centroid[0]:+.2f}, {centroid[1]:+.2f})")
        circles_list[i] = (float(centroid[0]), float(centroid[1]), r)

circles = circles_list

In [ ]:


# ── Schedules (found by Optuna search in star_curriculum.ipynb) ───────────────
N_ITERS = 12000

best_params = {
    "collision_delay": 0.27,
    "collision_ramp":  0.33,
    "exclusion_delay": 0.32,
    "exclusion_ramp":  0.47,
    "area_delay":      0.31,
    "area_ramp":       0.27,
    "perimeter_delay": 0.69,
    "perimeter_ramp":  0.45,
    "attraction_peak": 0.69,
    "attraction_ramp": 0.18,
}
term_schedules = make_term_schedules(best_params, N_ITERS)

# ── Optimize ──────────────────────────────────────────────────────────────────
snapshot_cb = SnapshotCallback(every=200)

results, circles_out, history, problem = optimize_multiple_radially_convex_sets_with_movable_circles(
    circles,
    sets,
    weight_area=1.0,
    weight_perimeter=2.0,
    weight_exclusion=10.0,
    weight_smoothness=2.0,
    weight_position_anchor=0.1,
    weight_circle_collision=50.0,
    weight_bounding_box=1.,
    weight_set_attraction=10.,
    circle_collision_alpha=0.5,
    offsets=offsets,
    term_schedules=term_schedules,
    optim_config=OptimConfig(n_iters=N_ITERS, learning_rate=0.005),
    callback=snapshot_cb,
)
print(f"Captured {len(snapshot_cb.snapshots)} snapshots")

In [ ]:
import matplotlib.cm as cm

fig, ax = plt.subplots(figsize=(14, 8))

cmap = cm.get_cmap("tab20", len(sets))

# Draw star-shaped boundaries (largest sets first so smaller ones appear on top)
order = sorted(range(len(sets)), key=lambda s: len(sets[s]), reverse=True)
for s in order:
    result = results[s]
    cx, cy = result["center"]
    radii = result["radii"]
    angles = result["angles"]
    bx = np.append(cx + radii * np.cos(angles), cx + radii[0] * np.cos(angles[0]))
    by = np.append(cy + radii * np.sin(angles), cy + radii[0] * np.sin(angles[0]))
    color = cmap(s)
    ax.fill(bx, by, alpha=0.10, color=color)
    ax.plot(bx, by, color=color, linewidth=1.0, alpha=0.7, label=set_names[s])
    ax.plot(cx, cy, "+", color=color, markersize=6, markeredgewidth=1.2)

circles_array = np.array(circles)
for i, ((ox, oy, r), (nx_, ny_, _)) in enumerate(zip(circles_array, circles_out)):
    node = leaves[i]
    ax.add_patch(plt.Circle((nx_, ny_), r, facecolor="steelblue", alpha=0.5,
                             edgecolor="steelblue", linewidth=1.0))
    ax.text(nx_, ny_, glotto_graph.nodes[node]["name"], fontsize=4,
            ha="center", va="center", color="k", fontweight="bold")

ax.set_aspect("equal")
ax.autoscale_view()
ax.margins(0.05)
ax.set_title("Language families — star-shaped set boundaries")
plt.tight_layout()

In [ ]:
from vizopt.animation import snapshots_to_animated_svg
from IPython.display import SVG

svg = snapshots_to_animated_svg(problem, snapshot_cb.snapshots, fps=10, size=800)
SVG(data=svg)

Next: dump optimized shapes to JSON and visualize in D3js

In [ ]:
ax = pd.DataFrame(history).set_index('iteration').plot(marker='.', lw=1, ms=3, figsize=(7, 3))
ax.set_ylabel('Loss value')
ax.set_title('Multi-set optimization history')
ax.set_yscale('linear')
ax.legend(bbox_to_anchor=(1.0, 1.0))
ax.set_ylim(0, 10000)
plt.tight_layout()

In [ ]:
import json, pathlib

data = {
    "circles": [
        {
            "id": leaves[i],
            "name": glotto_graph.nodes[leaves[i]]["name"],
            "wiki_name": glotto_graph.nodes[leaves[i]]["wikt_name"],
            "x": float(x),
            "y": float(y),
            "r": float(r),
        }
        for i, (x, y, r) in enumerate(circles_out)
    ],
    "sets": [
        {
            "name": set_names[s],
            "center": [float(results[s]["center"][0]), float(results[s]["center"][1])],
            "boundary": [
                [
                    float(results[s]["center"][0] + results[s]["radii"][k] * np.cos(results[s]["angles"][k])),
                    float(results[s]["center"][1] + results[s]["radii"][k] * np.sin(results[s]["angles"][k])),
                ]
                for k in range(len(results[s]["angles"]))
            ],
        }
        for s in range(len(sets))
    ],
}

out_path = pathlib.Path("data") / "optimized_layout.json"
out_path.parent.mkdir(exist_ok=True)
with open(out_path, "w") as f:
    json.dump(data, f)
print(f"Wrote {len(data['circles'])} circles and {len(data['sets'])} sets to {out_path}")


To-do:

* what languages do not have coordinates?
* better initialization for them, i.e. based on siblings etc.?
* visualize animation to understand spikes in loss
* implement smoother losses...
* loss for overall width and height